# SYNCEDSTRUCTURE
### **A Class to handle, crossreference and annotate Structural Proteomics Data**

Credits:
LICENCES:

In [1]:
### <DEPENDENCIES and SyncedStructure Methods display> 
import re, urllib.request, requests, io, copy, os, pickle
from typing import List, Optional, Dict, Any
from pyprojroot import here
from tqdm import tqdm

import pandas as pd
import seaborn as sns
import math, string
from functools import partial

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch, Rectangle
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from synced_structure import SyncedStructure
from viz3d import side_by_side
from exporter import export_panels_pdf

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

synced_structure_options = dir(SyncedStructure)
d = [x for x in synced_structure_options if x.startswith("__")]
h = [x for x in synced_structure_options
    if x.startswith("_") and not x.startswith("__")]
e = [x for x in synced_structure_options if not x.startswith("_")]
for g in (d, h, e):
  if g:
    [print(g[i : i + 4]) for i in range(0, len(g), 4)]
    print()
### <\DEPENDENCIES and SyncedStructure Methods display>

### <HELPERS>
def get_uniprot(uid):
    u = str(uid).strip().upper()
    if not re.match(r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})(-\d+)?$", u):
        raise ValueError()
    buf = io.StringIO(urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{u}.fasta").read().decode())
    return u, "".join(buf.read().splitlines()[1:])

def search_rcsb(
    organisms: Optional[List[str]] = None,
    keywords: Optional[List[str]] = None,
    title_contains: Optional[List[str]] = None,
    keywords_field: Optional[List[str]] = None,
    experimental_method: Optional[List[str]] = None,
    resolution_max: Optional[float] = None,
    resolution_min: Optional[float] = None,
    release_date_from: Optional[str] = None,
    release_date_to: Optional[str] = None,
    polymer_entity_count_min: Optional[int] = None,
    polymer_entity_count_max: Optional[int] = None,
    uniprot_ids: Optional[List[str]] = None,
    enzyme_classification: Optional[List[str]] = None,
    return_type: str = "entry",
    max_rows: int = 10000,
    return_all_hits: bool = False,
    include_computational: bool = False,
    sort_by: str = "score",
    sort_direction: str = "desc",
    timeout: int = 30,
) -> List[str]:
    """Search RCSB PDB using the Search API v2.

    Args:
        organisms: Scientific names matched with contains_phrase on
            rcsb_entity_source_organism.ncbi_scientific_name (OR).
        keywords: Free-text terms searched via full_text service (OR).
        title_contains: Words that must appear in struct.title (OR).
        keywords_field: Words that must appear in struct_keywords.pdbx_keywords (OR).
        experimental_method: Exact matches on exptl.method (OR).
        resolution_max: Upper resolution limit in Å.
        resolution_min: Lower resolution limit in Å.
        release_date_from: Earliest release date (YYYY-MM-DD).
        release_date_to: Latest release date (YYYY-MM-DD).
        polymer_entity_count_min: Minimum number of polymer entities.
        polymer_entity_count_max: Maximum number of polymer entities.
        uniprot_ids: UniProt accessions (OR).
        enzyme_classification: EC numbers (OR).
        return_type: One of entry, assembly, polymer_entity, non_polymer_entity,
            polymer_instance, mol_definition.
        max_rows: Maximum results to return (ignored if return_all_hits=True).
        return_all_hits: Return every matching identifier.
        include_computational: Also include Computed Structure Models.
        sort_by: Sort field (score, size, code, date, ...).
        sort_direction: asc or desc.
        timeout: Request timeout in seconds.

    Returns:
        List of matching identifiers.

    Raises:
        ValueError: If no search criteria are provided.
        requests.HTTPError: On non-2xx API response.
    """
    def t(attr, op, val, neg=False):
        return {"type": "terminal", "service": "text", "parameters": {"attribute": attr, "operator": op, "negation": neg, "value": val}}
    def ft(val):
        return {"type": "terminal", "service": "full_text", "parameters": {"value": val}}
    def g(nodes, op="or"):
        return nodes[0] if len(nodes) == 1 else {"type": "group", "logical_operator": op, "nodes": nodes}

    nodes = []
    if organisms:
        nodes.append(g([t("rcsb_entity_source_organism.ncbi_scientific_name", "contains_phrase", o) for o in organisms]))
    if keywords:
        nodes.append(g([ft(k) for k in keywords]))
    if title_contains:
        nodes.append(g([t("struct.title", "contains_words", x) for x in title_contains]))
    if keywords_field:
        nodes.append(g([t("struct_keywords.pdbx_keywords", "contains_words", x) for x in keywords_field]))
    if experimental_method:
        nodes.append(g([t("exptl.method", "exact_match", m) for m in experimental_method]))
    if resolution_min is not None or resolution_max is not None:
        nodes.append(t("rcsb_entry_info.resolution_combined", "range", {
            "from": resolution_min if resolution_min is not None else 0.0,
            "to": resolution_max if resolution_max is not None else 100.0,
            "include_lower": True, "include_upper": True}))
    if release_date_from or release_date_to:
        dv = {"include_lower": True, "include_upper": True}
        if release_date_from: dv["from"] = release_date_from
        if release_date_to: dv["to"] = release_date_to
        nodes.append(t("rcsb_accession_info.initial_release_date", "range", dv))
    if polymer_entity_count_min is not None or polymer_entity_count_max is not None:
        nodes.append(t("rcsb_entry_info.polymer_entity_count", "range", {
            "from": polymer_entity_count_min if polymer_entity_count_min is not None else 0,
            "to": polymer_entity_count_max if polymer_entity_count_max is not None else 9999,
            "include_lower": True, "include_upper": True}))
    if uniprot_ids:
        nodes.append(g([t("rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession", "exact_match", u) for u in uniprot_ids]))
    if enzyme_classification:
        nodes.append(g([t("rcsb_polymer_entity.rcsb_enzyme_class_combined.ec", "exact_match", e) for e in enzyme_classification]))
    if not nodes:
        raise ValueError("At least one search criterion must be provided")

    query = g(nodes, "and")
    content = ["experimental"] + (["computational"] if include_computational else [])
    opts: Dict[str, Any] = {"results_content_type": content, "sort": [{"sort_by": sort_by, "direction": sort_direction}], "scoring_strategy": "combined"}
    if return_all_hits:
        opts["return_all_hits"] = True
    else:
        opts["paginate"] = {"start": 0, "rows": max_rows}

    r = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json={"query": query, "return_type": return_type, "request_options": opts}, timeout=timeout)
    r.raise_for_status()
    return [h["identifier"] for h in r.json().get("result_set", [])]

### <\HELPERS>

['__class__', '__contains__', '__delattr__', '__dict__']
['__dir__', '__doc__', '__eq__', '__firstlineno__']
['__format__', '__ge__', '__getattribute__', '__getitem__']
['__getstate__', '__gt__', '__hash__', '__init__']
['__init_subclass__', '__iter__', '__le__', '__len__']
['__lt__', '__module__', '__ne__', '__new__']
['__reduce__', '__reduce_ex__', '__repr__', '__setattr__']
['__sizeof__', '__static_attributes__', '__str__', '__subclasshook__']
['__weakref__']

['_angles_overlay', '_apply_chain_options', '_atom_contacts', '_atom_pair_maps']
['_auth_of', '_block_entries', '_build_chains', '_ca_records']
['_chain_colors', '_chain_contact_partners', '_chain_description_map', '_chain_entity_map']
['_chain_geometry', '_chain_name', '_checked_matrix', '_clean_val']
['_compute_interface', '_compute_local_alignment', '_coords_touch', '_draw_pair']
['_entity_sequences', '_export', '_external_refs_for_entity', '_gray_matrix']
['_heavy_atoms', '_hydrophobicity', '_keep_mapped', '_label_of']
['_

# Antibody Interfaces over PDBs in Arboviruses

```
Signature:
search_rcsb(
    organisms: Optional[List[str]] = None,
    keywords: Optional[List[str]] = None,
    title_contains: Optional[List[str]] = None,
    keywords_field: Optional[List[str]] = None,
    experimental_method: Optional[List[str]] = None,
    resolution_max: Optional[float] = None,
    resolution_min: Optional[float] = None,
    release_date_from: Optional[str] = None,
    release_date_to: Optional[str] = None,
    polymer_entity_count_min: Optional[int] = None,
    polymer_entity_count_max: Optional[int] = None,
    uniprot_ids: Optional[List[str]] = None,
    enzyme_classification: Optional[List[str]] = None,
    return_type: str = 'entry',
    max_rows: int = 10000,
    return_all_hits: bool = False,
    include_computational: bool = False,
    sort_by: str = 'score',
    sort_direction: str = 'desc',
    timeout: int = 30,
) -> List[str]
Docstring:
Search RCSB PDB using the Search API v2.
```


In [2]:
refseq = get_uniprot('A0A142I5B9')
zika_ids = search_rcsb(
    organisms=["Zika virus"])
zika_abs = search_rcsb(
    organisms=["Zika virus"],
    keywords=["antibody", "Fab", "scFv", "mAb", "nanobody"])

In [3]:
print(f"ZIKA {len(zika_ids)}")
print(f"ZIKA and antibodies {len(zika_abs)}")
OUTDIR = here() / 'data/ZIKA-INTERFACES'
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"{refseq[0]}: {refseq[1][:60]}...")
print(zika_abs)

ZIKA 422
ZIKA and antibodies 66
A0A142I5B9: MKNPKKKSGGFRIVNMLKRGVARVSPFGGLKRLPAGLLLGHGPIRMVLAILAFLRFTAIK...
['8DV6', '7A3N', '5KVG', '5KVD', '5KVF', '8WNP', '8WNU', '8WO4', '6JEP', '7BSD', '5KVE', '5Y0A', '5UHY', '7CBP', '5GZR', '5VIG', '6DFI', '7A3U', '8JQM', '8JQN', '8JQS', '8CXG', '9XYB', '8JKF', '8JKH', '6UTA', '7C2T', '9PM6', '5LBV', '5JHL', '6NIU', '5GZN', '9OD2', '5OMZ', '5LBS', '7BUA', '5JHM', '8WO0', '5H32', '7BU8', '5LCV', '6PLK', '8EE5', '7BPK', '5GZO', '5H30', '9VWM', '8WN8', '7KCR', '9OWF', '6MID', '6LNT', '8CXH', '5H37', '9JS7', '8CXI', '6JFH', '6JFI', '9JS6', '8EE8', '8EED', '8EEE', '7BQ5', '7T17', '7YAR', '6NIP']


In [7]:
### <\SYNCED STRUCTURE LOOP>
zika_seq = refseq[1]
zika_id = refseq[0]
sync_summaries = []
OUT = OUTDIR / 'interfaces'
COV_CUTOFF = 5.0

for pdb_id in zika_abs:
    dir_pdb = OUT / pdb_id
    dir_pdb.mkdir(parents=True, exist_ok=True)
    model_outpath = dir_pdb / f"{pdb_id}_sync.pkl"
    if not model_outpath.exists():
        sync = SyncedStructure.from_rcsb_id(pdb_id)
        with open(model_outpath, 'wb') as f:
            pickle.dump(sync, f)
    else:
        with open(model_outpath, 'rb') as f:
            sync = pickle.load(f)
    biotypes = sync.moiety_summary
    non_protein = biotypes[biotypes['moiety'] != 'protein']['auth_chain_id']
    for notp in non_protein:
        sync.remove_chain(notp)
    zika_chains_df = sync.match_chains(zika_seq, min_identity=5)
    zika_chains = zika_chains_df[zika_chains_df['query_coverage_percent']>COV_CUTOFF]['auth_chain_id']
    for z in zika_chains:
        sync.set_reference_sequence(chain_id=z, sequence=zika_seq, remap=True)
        sync.remap_chain(z)
    sync_summary = sync.summary()
    sync_summary['pdb_id'] = pdb_id
    sync_summaries.append(sync_summary)

sync_df = pd.concat(sync_summaries, ignore_index=True)
print(sync_df.head())
fp_sync = OUT / 'sync_df_Zika_virus_antibodies.tsv'
if not fp_sync.exists():
    sync_df.to_csv(fp_sync, sep="\t")
    
### <\SYNCED STRUCTURE LOOP>

remapped A
remapped B
[PYTHON-INFO] 2026-10-18 22:31:55 | Loaded RCSB structure 7A3N | execution time: 0.94s | max memory: 431.1 MB
remapped A
[PYTHON-INFO] 2026-10-18 22:31:56 | Loaded RCSB structure 5KVG | execution time: 0.61s | max memory: 431.1 MB
[PYTHON-INFO] 2026-10-18 22:31:57 | Loaded RCSB structure 5KVD | execution time: 0.63s | max memory: 435.5 MB
[PYTHON-INFO] 2026-10-18 22:31:58 | Loaded RCSB structure 5KVF | execution time: 0.73s | max memory: 441.7 MB
[PYTHON-INFO] 2026-10-18 22:31:59 | Loaded RCSB structure 8WNP | execution time: 1.33s | max memory: 452.3 MB
remapped F
remapped E
[PYTHON-INFO] 2026-10-18 22:32:00 | Loaded RCSB structure 8WNU | execution time: 0.95s | max memory: 452.3 MB
remapped F
remapped E
[PYTHON-INFO] 2026-10-18 22:32:02 | Loaded RCSB structure 8WO4 | execution time: 0.97s | max memory: 452.3 MB
remapped F
remapped E
[PYTHON-INFO] 2026-10-18 22:32:03 | Loaded RCSB structure 6JEP | execution time: 1.05s | max memory: 452.3 MB
[PYTHON-INFO] 2026-10

In [9]:
import re
def _antibody_match():
    #case insensitive
    ['Fab','heavy','light','scfv']
    
for x in sync_df['description'].unique():
    print(x)

Envelope protein E
mAb Fab Heavy Chain
mAb Fab Light Chain
Core protein
EDE1 C10 Fab
ZIKA Envelope DIII
ZV-67 Antibody Fab Light Chain
ZV-67 Antibody Fab Heavy Chain
Zika Envelope DIII
ZV-2 Antibody Fab Light Chain
ZV-2 Antibody Fab Heavy Chain
Zika envelope DIII
ZV-64 Antibody Fab Light Chain
ZV-64 Antibody Fab Heavy Chain
VL Chains
VH Chains
Non-structural protein 1
Anti-fab nanobody
VL
VH
heavy chain of Fab ZK2B10
light chain of Fab ZK2B10
Genome polyprotein
1G5.3 Fab Heavy Chain
1G5.3 Fab Light Chain
NS1C
ZV-48 Antibody scFv
protein E
variable region of Fab ZKA190 heavy chain
variable region of Fab ZKA190 light chain
ZV67 Fab chain 1
ZV67 Fab chain 2
envelope protein
Small envelope protein M
Fab Heavy chain
Fab light chain
2-acetamido-2-deoxy-beta-D-glucopyranose-(1-4)-2-acetamido-2-deoxy-beta-D-glucopyranose
structural protein E
strutural protein M
Z23 Fab heavy chain
Z23 Fab light chain
Fab heavy chain
Zika virus envelope protein DIII
anti-Zika antibody Z021, Heavy Chain
anti-Zik